# HarvestLenz — Orange Grading Model Trainer

**Dataset:** [Citrus Fruits & Leaves — Mendeley DOI 10.17632/3f83gxmv57.2](https://data.mendeley.com/datasets/3f83gxmv57/2)

**Model:** MobileNetV2 (ImageNet pretrained) — matches the HarvestLenz `shared_mobilenet.py` architecture exactly.

**Output:** `orange.keras` — drop into `backend/backend/app/models/weights/` to enable CNN grading.

---

### Grade Mapping

| Citrus Disease | HarvestLenz Grade |
|---|---|
| healthy | Better |
| scab | Good |
| blackspot, canker, greening | Reject |

---

### Instructions

1. Runtime > Change runtime type > T4 GPU
2. Upload your Mendeley ZIP when Cell 2 prompts you
3. Runtime > Run all
4. Download orange.keras from Cell 11

In [ ]:
# Cell 1: Install dependencies and verify GPU
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'tensorflow>=2.13', 'scikit-learn', 'matplotlib',
     'seaborn', 'pillow', 'opencv-python-headless'],
    check=True
)

import tensorflow as tf
print('TensorFlow:', tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print('GPU ready:', gpus[0])
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: Upload Dataset ZIP
# Set UPLOAD_MODE = 'gdrive' if your ZIP is on Google Drive

UPLOAD_MODE     = 'upload'
GDRIVE_ZIP_PATH = '/content/drive/MyDrive/citrus_dataset.zip'

import os, zipfile, shutil
from pathlib import Path

EXTRACT_DIR = '/content/citrus_raw'

if UPLOAD_MODE == 'gdrive':
    from google.colab import drive
    drive.mount('/content/drive')
    zip_path = GDRIVE_ZIP_PATH
else:
    from google.colab import files as colab_files
    print('Upload the Mendeley citrus ZIP now...')
    uploaded = colab_files.upload()
    zip_path = list(uploaded.keys())[0]
    print(f'Uploaded: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)')

os.makedirs(EXTRACT_DIR, exist_ok=True)
print(f'Extracting to {EXTRACT_DIR}...')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

print('\nExtracted (top 3 levels):')
for root, dirs, fnames in os.walk(EXTRACT_DIR):
    depth = root.replace(EXTRACT_DIR, '').count(os.sep)
    if depth > 2:
        continue
    print('  ' * depth + os.path.basename(root) + '/')
    if depth == 2:
        print('  ' * (depth + 1) + f'[{len(fnames)} files]')

In [ ]:
# Cell 3: Map Disease Classes -> HarvestLenz Grades
import random
from collections import defaultdict

CLASS_MAP = {
    'healthy':    'Better',
    'scab':       'Good',
    'blackspot':  'Reject',
    'black spot': 'Reject',
    'canker':     'Reject',
    'greening':   'Reject',
    'melanose':   'Reject',
}

IMAGE_EXTS  = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
FRUIT_ONLY  = True
GRADES      = ['Better', 'Good', 'Reject']
DATASET_DIR = '/content/dataset_orange'

def classify_folder(name):
    n = name.lower().strip()
    for key, grade in CLASS_MAP.items():
        if n == key or n.startswith(key):
            return grade
    return None

def is_leaf_path(path_str):
    return any(p in path_str.lower() for p in ['leaf', 'leaves'])

for g in GRADES:
    os.makedirs(os.path.join(DATASET_DIR, g), exist_ok=True)

counts = defaultdict(int)
skipped = 0

for dirpath, _, filenames in os.walk(EXTRACT_DIR):
    grade = classify_folder(Path(dirpath).name)
    if grade is None:
        continue
    if FRUIT_ONLY and is_leaf_path(dirpath):
        skipped += sum(1 for f in filenames if Path(f).suffix.lower() in IMAGE_EXTS)
        continue
    for fname in filenames:
        if Path(fname).suffix.lower() not in IMAGE_EXTS:
            continue
        src = Path(dirpath) / fname
        stem = f"{src.stem}_{Path(dirpath).name}"
        dest = Path(DATASET_DIR) / grade / (stem + src.suffix)
        i = 1
        while dest.exists():
            dest = Path(DATASET_DIR) / grade / (stem + f'_{i}' + src.suffix)
            i += 1
        shutil.copy2(src, dest)
        counts[grade] += 1

total = sum(counts.values())
print('=== Dataset Summary ===')
for g in GRADES:
    bar = '#' * min(counts[g], 40)
    print(f'  {g:<8} {bar} {counts[g]}')
print(f'  Total: {total}')
if skipped:
    print(f'  Skipped {skipped} leaf images')

if total == 0:
    raise RuntimeError('No images found! Check your ZIP structure. Expected folders: healthy, scab, blackspot, canker, greening')

print('Dataset ready:', DATASET_DIR)

In [ ]:
# Cell 4: Citrus Color Augmentation
# Expands ~150 fruit images to ~400 per grade before splitting
import cv2
import numpy as np
from PIL import Image, ImageEnhance

TARGET_PER_GRADE = 400

PROFILES = [
    {'hue': -15, 'sat': 1.20, 'val': 1.05, 'bright': 1.08, 'contrast': 1.05},
    {'hue':  10, 'sat': 0.90, 'val': 0.95, 'bright': 0.95, 'contrast': 1.10},
    {'hue':  -8, 'sat': 1.10, 'val': 1.02, 'bright': 1.03, 'contrast': 1.00},
    {'hue':   5, 'sat': 0.80, 'val': 1.10, 'bright': 1.05, 'contrast': 0.95},
    {'hue': -20, 'sat': 1.15, 'val': 0.95, 'bright': 0.98, 'contrast': 1.08},
    {'hue':   0, 'sat': 0.70, 'val': 1.15, 'bright': 1.10, 'contrast': 0.90},
]

def augment_image(img_bgr, profile):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 0] = (hsv[:, :, 0] + profile['hue']) % 180
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * profile['sat'], 0, 255)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * profile['val'], 0, 255)
    result = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    pil = Image.fromarray(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    pil = ImageEnhance.Brightness(pil).enhance(profile['bright'])
    pil = ImageEnhance.Contrast(pil).enhance(profile['contrast'])
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

random.seed(42)

for grade in GRADES:
    gdir = os.path.join(DATASET_DIR, grade)
    originals = [f for f in os.listdir(gdir)
                 if Path(f).suffix.lower() in IMAGE_EXTS and not f.startswith('aug_')]
    n_orig = len(originals)
    if n_orig == 0:
        print(f'  {grade}: no images, skipping')
        continue
    needed = max(0, TARGET_PER_GRADE - n_orig)
    if needed == 0:
        print(f'  {grade}: {n_orig} images, no augmentation needed')
        continue
    pool = (originals * (needed // n_orig + 2))[:needed]
    random.shuffle(pool)
    generated = 0
    for i, fname in enumerate(pool):
        img = cv2.imread(os.path.join(gdir, fname))
        if img is None:
            continue
        aug = augment_image(img, PROFILES[i % len(PROFILES)])
        cv2.imwrite(os.path.join(gdir, f'aug_{i:04d}_{Path(fname).stem}.jpg'), aug)
        generated += 1
    total_now = len([f for f in os.listdir(gdir) if Path(f).suffix.lower() in IMAGE_EXTS])
    print(f'  {grade}: {n_orig} orig + {generated} aug = {total_now} total')

print('Augmentation complete')

In [ ]:
# Cell 5: Train / Val / Test Splits (80 / 10 / 10)
SPLIT_DIR   = '/content/orange_split'
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10

random.seed(42)
split_summary = {}

for grade in GRADES:
    gdir = os.path.join(DATASET_DIR, grade)
    if not os.path.isdir(gdir):
        continue
    imgs = [f for f in os.listdir(gdir) if Path(f).suffix.lower() in IMAGE_EXTS]
    random.shuffle(imgs)
    n       = len(imgs)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    splits  = {
        'train': imgs[:n_train],
        'val':   imgs[n_train:n_train + n_val],
        'test':  imgs[n_train + n_val:],
    }
    split_summary[grade] = {k: len(v) for k, v in splits.items()}
    for sname, simgs in splits.items():
        ddir = os.path.join(SPLIT_DIR, sname, grade)
        os.makedirs(ddir, exist_ok=True)
        for img_name in simgs:
            dst = os.path.join(ddir, img_name)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(gdir, img_name), dst)

print(f'{"Grade":<10} {"Train":>7} {"Val":>7} {"Test":>7} {"Total":>7}')
print('-' * 36)
for grade, s in split_summary.items():
    t = sum(s.values())
    print(f'{grade:<10} {s["train"]:>7} {s["val"]:>7} {s["test"]:>7} {t:>7}')
print('Splits created at:', SPLIT_DIR)

In [ ]:
# Cell 6: Build MobileNetV2 Model
# Identical architecture to HarvestLenz shared_mobilenet.py build_grading_model()
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, Model, Input

IMG_SIZE    = 224
NUM_CLASSES = 3
BATCH_SIZE  = 16
CLASSES     = ['Better', 'Good', 'Reject']

def build_model(num_classes=3):
    base = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    inp = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    mdl = Model(inp, out)
    mdl._base_model = base
    return mdl

model = build_model(NUM_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()
trainable = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
print(f'Trainable parameters (head only): {trainable:,}')

In [ ]:
# Cell 7: Data Generators + Class Weights
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

TRAIN_DIR = os.path.join(SPLIT_DIR, 'train')
VAL_DIR   = os.path.join(SPLIT_DIR, 'val')
TEST_DIR  = os.path.join(SPLIT_DIR, 'test')

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    brightness_range=(0.7, 1.3),
    zoom_range=[0.85, 1.15],
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=8,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest',
)
val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

def make_flow(gen, directory, shuffle=True):
    return gen.flow_from_directory(
        directory,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=shuffle,
    )

train_flow = make_flow(train_gen, TRAIN_DIR, shuffle=True)
val_flow   = make_flow(val_gen,   VAL_DIR,   shuffle=False)
test_flow  = make_flow(val_gen,   TEST_DIR,  shuffle=False)

print('Class indices :', train_flow.class_indices)
print('Train samples :', train_flow.samples)
print('Val samples   :', val_flow.samples)
print('Test samples  :', test_flow.samples)

labels    = train_flow.classes
cw_values = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weights = {i: float(w) for i, w in enumerate(cw_values)}
print('Class weights :', class_weights)

In [ ]:
# Cell 8: Phase 1 — Head Training (base frozen, 10 epochs)
OUTPUT_DIR    = '/content/orange_weights'
PHASE1_EPOCHS = 10
os.makedirs(OUTPUT_DIR, exist_ok=True)

ckpt_p1 = os.path.join(OUTPUT_DIR, 'orange_phase1_best.keras')

print('PHASE 1: Head training — MobileNetV2 base frozen')
print('=' * 52)

callbacks_p1 = [
    tf.keras.callbacks.ModelCheckpoint(
        ckpt_p1, monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1
    ),
]

hist1 = model.fit(
    train_flow,
    epochs=PHASE1_EPOCHS,
    validation_data=val_flow,
    class_weight=class_weights,
    callbacks=callbacks_p1,
)

_, val_acc_p1 = tf.keras.models.load_model(ckpt_p1).evaluate(val_flow, verbose=0)
print(f'Phase 1 best val accuracy: {val_acc_p1:.4f}')

In [ ]:
# Cell 9: Phase 2 — Fine-tune top MobileNetV2 layers (20 epochs)
PHASE2_EPOCHS  = 20
FINE_TUNE_FROM = 100

ckpt_p2  = os.path.join(OUTPUT_DIR, 'orange_phase2_best.keras')
ft_model = tf.keras.models.load_model(ckpt_p1)

backbone = ft_model.layers[1]
backbone.trainable = True
for layer in backbone.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

print(f'PHASE 2: Fine-tuning layers {FINE_TUNE_FROM}..{len(backbone.layers)} of MobileNetV2')
print('=' * 52)

ft_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    tf.keras.callbacks.ModelCheckpoint(
        ckpt_p2, monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1
    ),
]

hist2 = ft_model.fit(
    train_flow,
    epochs=PHASE1_EPOCHS + PHASE2_EPOCHS,
    initial_epoch=PHASE1_EPOCHS,
    validation_data=val_flow,
    class_weight=class_weights,
    callbacks=callbacks_p2,
)

print('Phase 2 complete')

In [ ]:
# Cell 10: Pick Best Model + Evaluate
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

m_p1 = tf.keras.models.load_model(ckpt_p1)
_, acc_p1 = m_p1.evaluate(test_flow, verbose=0)
print(f'Phase 1 test accuracy: {acc_p1:.4f}')

acc_p2 = 0.0
if os.path.exists(ckpt_p2):
    m_p2 = tf.keras.models.load_model(ckpt_p2)
    _, acc_p2 = m_p2.evaluate(test_flow, verbose=0)
    print(f'Phase 2 test accuracy: {acc_p2:.4f}')

best_model = m_p2 if acc_p2 > acc_p1 else m_p1
winner     = 'Phase 2' if acc_p2 > acc_p1 else 'Phase 1'
print(f'{winner} wins with test accuracy: {max(acc_p1, acc_p2):.4f}')

FINAL_PATH = os.path.join(OUTPUT_DIR, 'orange.keras')
best_model.save(FINAL_PATH)
size_mb = os.path.getsize(FINAL_PATH) / 1024 / 1024
print(f'Saved: {FINAL_PATH} ({size_mb:.1f} MB)')

# Confusion matrix
test_flow.reset()
y_true     = test_flow.classes
y_pred     = best_model.predict(test_flow, verbose=1)
y_pred_cls = np.argmax(y_pred, axis=1)

print('\nClassification Report:')
print(classification_report(y_true, y_pred_cls, target_names=CLASSES))

cm = confusion_matrix(y_true, y_pred_cls)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

all_acc     = hist1.history['accuracy']     + hist2.history.get('accuracy', [])
all_val_acc = hist1.history['val_accuracy'] + hist2.history.get('val_accuracy', [])
axes[1].plot(all_acc,     label='Train',      color='#FF8C00', linewidth=2)
axes[1].plot(all_val_acc, label='Validation', color='#FF4500', linewidth=2, linestyle='--')
axes[1].axvline(PHASE1_EPOCHS - 1, color='gray', linestyle=':', alpha=0.7, label='Phase 1->2')
axes[1].set_title('Training Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Orange Grading Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'orange_results.png'), dpi=150, bbox_inches='tight')
plt.show()

all_loss     = hist1.history['loss']     + hist2.history.get('loss', [])
all_val_loss = hist1.history['val_loss'] + hist2.history.get('val_loss', [])
with open(os.path.join(OUTPUT_DIR, 'orange_history.json'), 'w') as f:
    json.dump({
        'accuracy':     [float(x) for x in all_acc],
        'val_accuracy': [float(x) for x in all_val_acc],
        'loss':         [float(x) for x in all_loss],
        'val_loss':     [float(x) for x in all_val_loss],
    }, f, indent=2)

In [ ]:
# Cell 11: Download orange.keras
from google.colab import files as colab_files

print('Output files:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    mb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024 / 1024
    print(f'  {fname:<40} {mb:.1f} MB')

print('\nDownloading orange.keras...')
colab_files.download(FINAL_PATH)

print('')
print('DONE!')
print('Next steps:')
print('  1. Copy orange.keras to:')
print('     backend/backend/app/models/weights/orange.keras')
print('  2. Restart the HarvestLenz backend')
print('  3. Orange grading upgrades from heuristic to CNN automatically')